# 07. Komuter stations (Version 6, step 2)

Where is each Komuter station? This notebook gives every station in `komuter_stations.csv` its coordinates from the **KTMB GTFS** feed, fixes the stations GTFS gets wrong or misses, checks the Klang Valley list against the district boundaries, and finds the nearest Rapid Rail station (to mark interchanges).

| Section | What it does |
|---|---|
| 0 | Settings |
| 1 | Load the station list and the KTMB GTFS stops |
| 2 | Match names |
| 3 | Fix stations that GTFS misses or gets wrong (OpenStreetMap lookup) |
| 4 | Check regions against the district boundaries |
| 5 | Nearest Rapid Rail station and interchanges |
| 6 | Quick map check |
| 7 | Save the outputs |

**Known GTFS problems (found 1 October 2026):** Seri Setia has the same coordinates as Setia Jaya; Segambut Utara (opened March 2026) is missing; "Segambut 2" in the ridership data is Segambut Utara under a temporary name for its first two days.

Needs `data/raw/gtfs_ktmb.zip` (from https://api.data.gov.my/gtfs-static/ktmb), and from earlier notebooks `komuter_stations.csv`, `districts.geojson` and `stations.geojson` in `data/processed`.

## 0. Settings

In [9]:
from pathlib import Path
import json, re, zipfile
from datetime import datetime
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

GTFS_ZIP = Path("data/raw/gtfs_ktmb.zip")
PROCESSED = Path("data/processed")
OSM_CACHE = Path("data/raw/osm_komuter_fixes.csv")   # OpenStreetMap lookups, saved so they run once

INTERCHANGE_M = 450               # <- a Rapid Rail station within this distance counts as an interchange
OSM_SEARCH_M = 3000               # <- search radius around the fallback point for the OpenStreetMap lookup
USE_OSM = True                    # <- False = skip OpenStreetMap and use the fallback points

# Ridership name: GTFS stop_name, where the two differ beyond capitals and punctuation
GTFS_NAMES = {
    "Bandar Tasek Selatan": "BDR TASEK SELATAN",
    "Pelabuhan Klang Selatan": "PEL KLANG SEL",
    "Jalan Kastam": "JLN KASTAM",
    "Kampung Dato Harun": "KG DATO HARUN",
    "Kampung Raja Uda": "KG RAJA UDA",
    "Pulau Sebang (Tampin)": "PULAU SEBANG/TAMPIN",
    "Midvalley": "PERHENTIAN MIDVALLEY",
    "Batu Kentonmen": "BATU KENTOMENN",      # GTFS spelling
}

# Ridership names that are another station under a temporary name
SAME_AS = {"Segambut 2": "Segambut Utara"}   # <- first two days after opening (3 and 4 March 2026)

# Stations to look up in OpenStreetMap, with a fallback point (lat, lon) if the lookup fails.
# Fallbacks are approximate: midway between the neighbouring stations on the line.
FIX = {
    "Seri Setia":     {"osm_name": "Seri Setia",     "fallback": (3.0841, 101.6219)},   # between Kg Dato Harun and Setia Jaya
    "Segambut Utara": {"osm_name": "Segambut Utara", "fallback": (3.1948, 101.6507)},   # between Segambut and Kepong
}

## 1. Load the station list and the KTMB GTFS stops

The GTFS file covers all of KTMB (156 stops, including intercity and ETS stations); only the Komuter stations are used.

In [10]:
stations = pd.read_csv(PROCESSED / "komuter_stations.csv")[["station", "region", "district", "corridor"]]   # base columns, so the notebook can be re-run
with zipfile.ZipFile(GTFS_ZIP) as z:
    with z.open("stops.txt") as f:
        stops = pd.read_csv(f)
stops["stop_lat"] = stops["stop_lat"].astype(float)
stops["stop_lon"] = stops["stop_lon"].astype(float)
print(f"{len(stations)} Komuter stations in the ridership data; {len(stops)} KTMB GTFS stops")

67 Komuter stations in the ridership data; 156 KTMB GTFS stops


## 2. Match names

Names are compared in capitals with punctuation removed; the `GTFS_NAMES` list in Section 0 covers the rest. Stations in `SAME_AS` take the position of the station they are the same as.

In [11]:
def clean(s):
    return re.sub(r"[^A-Z0-9 ]", " ", str(s).upper()).split()

stops["key"] = stops["stop_name"].map(lambda s: " ".join(clean(s)))
by_key = stops.drop_duplicates("key").set_index("key")

def find(name):
    target = GTFS_NAMES.get(name, name)
    k = " ".join(clean(target))
    if k in by_key.index:
        r = by_key.loc[k]
        return pd.Series({"gtfs_name": r["stop_name"], "lat": r["stop_lat"], "lon": r["stop_lon"]})
    return pd.Series({"gtfs_name": None, "lat": None, "lon": None})

stations = stations.join(stations["station"].map(lambda n: find(SAME_AS.get(n, n))).apply(pd.Series))
stations["source"] = stations["gtfs_name"].notna().map({True: "GTFS", False: None})

unmatched = stations.loc[stations["gtfs_name"].isna(), "station"].tolist()
print(f"Matched in GTFS: {stations['gtfs_name'].notna().sum()} of {len(stations)}")
print("Not in GTFS:", unmatched if unmatched else "none")

# Two stations at exactly the same point usually means a copying mistake in GTFS
dup = stations[stations["gtfs_name"].notna()].groupby(["lat", "lon"])["station"].apply(list)
dup = dup[dup.map(len) > 1]
dup = [d for d in dup if not set(d) & set(SAME_AS)]   # a station and its temporary name may share a point
print("Different stations at the same point:", dup if dup else "none")

Matched in GTFS: 65 of 67
Not in GTFS: ['Segambut 2', 'Segambut Utara']
Different stations at the same point: [['Seri Setia', 'Setia Jaya']]


## 3. Fix stations that GTFS misses or gets wrong

Each station in `FIX` is looked up in OpenStreetMap: railway stations within `OSM_SEARCH_M` of the fallback point whose name contains the station name. The result is saved to `data/raw/osm_komuter_fixes.csv`, so later runs do not ask OpenStreetMap again (delete the file to look up again). If nothing is found, the fallback point is used and marked **approximate**.

In [12]:
def osm_lookup(name, lat, lon):
    import osmnx as ox
    tags = {"railway": ["station", "halt", "stop"], "public_transport": ["station", "stop_position"]}
    g = ox.features_from_point((lat, lon), tags=tags, dist=OSM_SEARCH_M)
    if g.empty or "name" not in g.columns:
        return None
    g = g[g["name"].fillna("").str.contains(name, case=False)]
    if g.empty:
        return None
    c = g.to_crs(3857).geometry.centroid.to_crs(4326).iloc[0]   # first match, as a point
    return round(c.y, 6), round(c.x, 6), g["name"].iloc[0]

cache = pd.read_csv(OSM_CACHE) if OSM_CACHE.exists() else pd.DataFrame(columns=["station", "lat", "lon", "osm_name_found"])
new_rows = []
for st, f in FIX.items():
    row = cache[cache["station"] == st]
    if row.empty and USE_OSM:
        try:
            res = osm_lookup(f["osm_name"], *f["fallback"])
        except Exception as e:
            print(f"{st}: OpenStreetMap lookup failed ({type(e).__name__}: {str(e)[:80]})")
            res = None
        if res:
            new_rows.append({"station": st, "lat": res[0], "lon": res[1], "osm_name_found": res[2]})
            row = pd.DataFrame([new_rows[-1]])
    if not row.empty:
        lat, lon, how = row["lat"].iloc[0], row["lon"].iloc[0], f"OpenStreetMap ({row['osm_name_found'].iloc[0]})"
        src = "OpenStreetMap"
    else:
        (lat, lon), how, src = f["fallback"], "fallback point, approximate", "approximate"
    for s in [st] + [k for k, v in SAME_AS.items() if v == st]:
        stations.loc[stations["station"] == s, ["lat", "lon", "source"]] = [lat, lon, src]
    print(f"{st}: {lat:.5f}, {lon:.5f} from {how}")

if new_rows:
    pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True).to_csv(OSM_CACHE, index=False)

still = stations.loc[stations["lat"].isna(), "station"].tolist()
print("\nStations still without coordinates:", still if still else "none")

Seri Setia: 3.08453, 101.62190 from OpenStreetMap (Seri Setia)
Segambut Utara: 3.19202, 101.65514 from OpenStreetMap (KA05A Segambut Utara)

Stations still without coordinates: none


## 4. Check regions against the district boundaries

Each station is placed in one of the seven Klang Valley districts from `districts.geojson`, if it lies inside one. The typed region from `07komuter1` is then compared: any station marked **Klang Valley** that lies outside the districts, or the other way round, is listed. Districts outside the Klang Valley keep the typed name.

In [13]:
pts = gpd.GeoDataFrame(stations, geometry=[Point(xy) for xy in zip(stations["lon"], stations["lat"])], crs=4326)
districts = gpd.read_file(PROCESSED / "districts.geojson").to_crs(4326)
pts = gpd.sjoin(pts, districts[["district", "geometry"]].rename(columns={"district": "kv_district"}),
                how="left", predicate="within").drop(columns="index_right")
pts["in_kv_districts"] = pts["kv_district"].notna()

check = pts[(pts["region"] == "Klang Valley") != pts["in_kv_districts"]]
if check.empty:
    print("All typed regions agree with the district boundaries.")
else:
    print("Typed region and district boundaries disagree:")
    print(check[["station", "region", "district", "kv_district", "source"]].to_string(index=False))

same_name = lambda a, b: str(a).replace("W.P. ", "") == str(b).replace("W.P. ", "")   # "Kuala Lumpur" = "W.P. Kuala Lumpur"
wrong_district = pts[pts["in_kv_districts"] & ~pts.apply(lambda r: same_name(r["district"], r["kv_district"]), axis=1)]
if len(wrong_district):
    print("\nTyped district corrected from the boundaries:")
    print(wrong_district[["station", "district", "kv_district"]].to_string(index=False))
pts.loc[pts["in_kv_districts"], "district"] = pts["kv_district"]

All typed regions agree with the district boundaries.


## 5. Nearest Rapid Rail station and interchanges

For each Komuter station, the nearest station on the main map (`stations.geojson`) and the walking-free straight-line distance. Within `INTERCHANGE_M` it counts as an **interchange**: those stations link the two pages, and Step 3 compares their Komuter hours with their Rapid Rail station type.

In [14]:
rr = gpd.read_file(PROCESSED / "stations.geojson").to_crs(3857)
p = pts.to_crs(3857)
near = gpd.sjoin_nearest(p[["station", "geometry"]], rr[["name", "key", "geometry"]], how="left", distance_col="rr_distance_m")
near = near.drop_duplicates("station").set_index("station")
pts["rr_station"] = pts["station"].map(near["name"])
pts["rr_key"] = pts["station"].map(near["key"])
pts["rr_distance_m"] = pts["station"].map(near["rr_distance_m"]).round(0)
pts["interchange"] = pts["rr_distance_m"] <= INTERCHANGE_M

ic = pts[pts["interchange"]].sort_values("rr_distance_m")
print(f"Interchanges (Rapid Rail station within {INTERCHANGE_M} m): {len(ic)}")
print(ic[["station", "rr_station", "rr_distance_m"]].to_string(index=False))
close = pts[~pts["interchange"] & (pts["rr_distance_m"] <= 2 * INTERCHANGE_M)]
if len(close):
    print(f"\nJust outside the limit (up to {2 * INTERCHANGE_M} m), worth a look:")
    print(close[["station", "rr_station", "rr_distance_m"]].to_string(index=False))

Interchanges (Rapid Rail station within 450 m): 13
             station           rr_station  rr_distance_m
      Abdullah Hukum       Abdullah Hukum            6.0
Bandar Tasek Selatan Bandar Tasik Selatan           19.0
              Kajang               Kajang           66.0
        Kampung Batu         Kampung Batu           71.0
         Subang Jaya          Subang Jaya           85.0
      Kepong Sentral  Sri Damansara Timur           95.0
          KL Sentral           KL Sentral          119.0
         Bank Negara            Bandaraya          161.0
        Sungai Buloh         Sungai Buloh          184.0
               Putra                 PWTC          296.0
      Batu Kentonmen             Kentomen          356.0
        Kuala Lumpur           Pasar Seni          401.0
       Salak Selatan        Salak Selatan          440.0

Just outside the limit (up to 900 m), worth a look:
    station     rr_station  rr_distance_m
     Sentul   Sentul Barat          652.0
  Midvalley Ab

## 6. Quick map check

A plain map to spot anything out of place: Komuter stations coloured by region, with Rapid Rail stations in grey. Approximate positions are drawn with a black ring. Hover for names.

In [15]:
import folium

REGION_COLOURS = {"Klang Valley": "#0C2363", "Outer belt": "#30BCF8", "Other": "#999999"}   # <- check-map colours only

kv_pts = pts[pts["region"] != "Other"]
m = folium.Map(location=[kv_pts["lat"].mean(), kv_pts["lon"].mean()], zoom_start=10, tiles="OpenStreetMap")
folium.GeoJson(districts, style_function=lambda f: {"color": "#666", "weight": 1, "fillOpacity": 0}).add_to(m)
for _, r in rr.to_crs(4326).iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=2, color="#bbb", fill=True, tooltip=r["name"]).add_to(m)
for _, r in pts.iterrows():
    approx = r["source"] == "approximate"
    folium.CircleMarker([r["lat"], r["lon"]], radius=5, color="#000" if approx else REGION_COLOURS[r["region"]],
                        weight=2 if approx else 1, fill=True, fill_color=REGION_COLOURS[r["region"]], fill_opacity=0.9,
                        tooltip=f"{r['station']} ({r['region']}, {r['source']})").add_to(m)
m

## 7. Save the outputs

- `komuter_stations.geojson`: every station with region, district, corridor, coordinates and their source, nearest Rapid Rail station, distance and interchange flag. `same_as` names the station a temporary name belongs to.
- `komuter_stations.csv` is updated with the same columns (without the geometry).

In [16]:
pts["same_as"] = pts["station"].map(SAME_AS)
cols = ["station", "same_as", "region", "district", "corridor", "lat", "lon", "source",
        "rr_station", "rr_key", "rr_distance_m", "interchange"]
out = pts[cols + ["geometry"]].copy()
out.to_file(PROCESSED / "komuter_stations.geojson", driver="GeoJSON")
out.drop(columns="geometry").to_csv(PROCESSED / "komuter_stations.csv", index=False)

meta_path = PROCESSED / "komuter_metadata.json"
meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
meta.update({
    "stations_created": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "station_coordinates": "KTMB GTFS (data.gov.my, CC BY 4.0); fixes from OpenStreetMap or approximate points",
    "station_sources": out["source"].value_counts().to_dict(),
    "interchange_m": INTERCHANGE_M,
    "interchanges": int(out["interchange"].sum()),
    "same_as": SAME_AS,
})
meta_path.write_text(json.dumps(meta, indent=2))
print(out["source"].value_counts().to_string())
for f in ["komuter_stations.geojson", "komuter_stations.csv", "komuter_metadata.json"]:
    print(f"{f}: {(PROCESSED / f).stat().st_size / 1e3:.1f} kB")

source
GTFS             64
OpenStreetMap     3
komuter_stations.geojson: 26.3 kB
komuter_stations.csv: 7.5 kB
komuter_metadata.json: 0.9 kB


___
**END**